# ThreadCraft — Garment Classifier · Step 2: Training

Fine-tunes a Vision Transformer to predict garment type from a product/reference image, so ThreadCraft can auto-suggest a cloth type when a customer uploads a reference photo at wizard Step 2.

**Before running:**
1. `01_data_cleaning.ipynb` must have completed and pushed `<your-username>/threadcraft-garments-cleaned`
2. Kaggle sidebar → **Accelerator: GPU T4 x2**, **Internet: On**
3. Kaggle → Add-ons → Secrets → `HF_TOKEN` attached
4. Run via **Save & Run All (Commit)**, not interactively — an interactive session dies when you close the tab and you lose the GPU hours

**Expected runtime:** roughly 25–50 minutes for 4 epochs, depending on class count and image throughput.

> **T4 note:** T4 supports **fp16** but not bf16. `fp16=True` below is deliberate — switching it to `bf16=True` will fail on this GPU.

In [ ]:
# ── CONFIG ────────────────────────────────────────────────────────────────
HF_USERNAME = "your-hf-username"  # <-- CHANGE THIS (must match 01_data_cleaning.ipynb)

DATASET_REPO_ID = f"{HF_USERNAME}/threadcraft-garments-cleaned"
MODEL_REPO_ID = f"{HF_USERNAME}/threadcraft-garment-classifier"

BASE_MODEL = "google/vit-base-patch16-224-in21k"

NUM_EPOCHS = 4
BATCH_SIZE = 64          # per device; drop to 32 if you hit CUDA OOM
LEARNING_RATE = 3e-4
WEIGHT_DECAY = 0.01
WARMUP_RATIO = 0.1
RANDOM_SEED = 42
PUSH_TO_HUB = True

OUTPUT_DIR = "/kaggle/working/garment-classifier"

In [ ]:
!pip install -q -U transformers datasets huggingface_hub scikit-learn accelerate evaluate

In [ ]:
import torch

print(f"torch {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    for i in range(torch.cuda.device_count()):
        print(f"  GPU {i}: {torch.cuda.get_device_name(i)}")
else:
    print("NO GPU — set Accelerator to 'GPU T4 x2' in the Kaggle sidebar, then restart the session.")

In [ ]:
import os

from huggingface_hub import login

HF_TOKEN = None
try:
    from kaggle_secrets import UserSecretsClient

    HF_TOKEN = UserSecretsClient().get_secret("HF_TOKEN")
except Exception as e:
    print(f"Not on Kaggle or secret missing ({e}). Falling back to the HF_TOKEN env var.")
    HF_TOKEN = os.environ.get("HF_TOKEN")

if HF_TOKEN:
    login(token=HF_TOKEN)
    print("Logged in to Hugging Face.")
elif PUSH_TO_HUB:
    raise RuntimeError("No HF_TOKEN available but PUSH_TO_HUB is True.")

## 1. Load the cleaned dataset

In [ ]:
from datasets import load_dataset

ds = load_dataset(DATASET_REPO_ID)
print(ds)

labels = ds["train"].features["label"].names
label2id = {name: i for i, name in enumerate(labels)}
id2label = {i: name for name, i in label2id.items()}
NUM_LABELS = len(labels)
print(f"\n{NUM_LABELS} classes: {labels}")

## 2. Image preprocessing

`set_transform` applies preprocessing **lazily, per batch**. Doing it eagerly with `.map()` would materialise every image as a 224×224×3 float tensor — roughly 600 KB each, tens of GB across the dataset — and blow past Kaggle's ~29 GB RAM.

Light augmentation on train only (horizontal flip, small rotation). Garment photos are near-canonical product shots, so aggressive augmentation mostly adds noise; validation and test get deterministic resize + normalise only.

In [ ]:
from torchvision.transforms import (
    Compose,
    Normalize,
    RandomHorizontalFlip,
    RandomRotation,
    Resize,
    ToTensor,
)
from transformers import AutoImageProcessor

processor = AutoImageProcessor.from_pretrained(BASE_MODEL)
image_size = processor.size["height"]
normalize = Normalize(mean=processor.image_mean, std=processor.image_std)
print(f"Base model expects {image_size}x{image_size}; source images are 60x80 (upscaled).")

train_tf = Compose(
    [
        Resize((image_size, image_size)),
        RandomHorizontalFlip(p=0.5),
        RandomRotation(degrees=8),
        ToTensor(),
        normalize,
    ]
)
eval_tf = Compose([Resize((image_size, image_size)), ToTensor(), normalize])


def _apply(transform):
    def fn(batch):
        # .convert("RGB") guards against any greyscale/RGBA image in the set —
        # a single 1-channel image would otherwise crash the batch collation.
        batch["pixel_values"] = [transform(img.convert("RGB")) for img in batch["image"]]
        return batch

    return fn


ds["train"].set_transform(_apply(train_tf))
ds["validation"].set_transform(_apply(eval_tf))
ds["test"].set_transform(_apply(eval_tf))

example = ds["train"][0]
print(f"pixel_values shape: {example['pixel_values'].shape}, label: {id2label[example['label']]}")

## 3. Model, metrics, trainer

In [ ]:
from transformers import AutoModelForImageClassification

model = AutoModelForImageClassification.from_pretrained(
    BASE_MODEL,
    num_labels=NUM_LABELS,
    id2label=id2label,
    label2id=label2id,
    # The pretrained head is for ImageNet-21k's 21,843 classes; we replace it.
    ignore_mismatched_sizes=True,
)
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Trainable parameters: {trainable:,}")

In [ ]:
import numpy as np
from sklearn.metrics import accuracy_score, f1_score


def compute_metrics(eval_pred):
    logits, y_true = eval_pred
    y_pred = np.argmax(logits, axis=-1)
    return {
        "accuracy": accuracy_score(y_true, y_pred),
        # macro_f1 is the headline number for this dataset: accuracy is inflated
        # by the dominant Tshirts/Shirts classes, macro-F1 weights every garment
        # type equally and so exposes failure on the rarer ones.
        "macro_f1": f1_score(y_true, y_pred, average="macro", zero_division=0),
        "weighted_f1": f1_score(y_true, y_pred, average="weighted", zero_division=0),
    }


def collate_fn(examples):
    import torch as _torch

    return {
        "pixel_values": _torch.stack([e["pixel_values"] for e in examples]),
        "labels": _torch.tensor([e["label"] for e in examples]),
    }

In [ ]:
import inspect

from transformers import Trainer, TrainingArguments

args_kwargs = dict(
    output_dir=OUTPUT_DIR,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE,
    num_train_epochs=NUM_EPOCHS,
    learning_rate=LEARNING_RATE,
    weight_decay=WEIGHT_DECAY,
    warmup_ratio=WARMUP_RATIO,
    fp16=torch.cuda.is_available(),  # T4 supports fp16, NOT bf16
    logging_steps=50,
    save_total_limit=1,
    load_best_model_at_end=True,
    metric_for_best_model="macro_f1",
    greater_is_better=True,
    remove_unused_columns=False,  # required: we need the raw `image` column for set_transform
    seed=RANDOM_SEED,
    report_to="none",
    dataloader_num_workers=2,
)

# transformers renamed evaluation_strategy -> eval_strategy in v4.46.
# Pick whichever this version accepts so the notebook survives the rename.
_params = inspect.signature(TrainingArguments.__init__).parameters
_strategy_key = "eval_strategy" if "eval_strategy" in _params else "evaluation_strategy"
args_kwargs[_strategy_key] = "epoch"
args_kwargs["save_strategy"] = "epoch"
print(f"Using '{_strategy_key}' for the evaluation strategy argument.")

training_args = TrainingArguments(**args_kwargs)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=ds["train"],
    eval_dataset=ds["validation"],
    data_collator=collate_fn,
    compute_metrics=compute_metrics,
)

## 4. Train

In [ ]:
train_result = trainer.train()
print(train_result.metrics)

In [ ]:
# Save locally FIRST, before any Hub interaction — if the push fails, the
# trained weights still survive in /kaggle/working across the commit.
trainer.save_model(OUTPUT_DIR)
processor.save_pretrained(OUTPUT_DIR)
print(f"Model + processor saved to {OUTPUT_DIR}")

## 5. Evaluate on the held-out test split

The test split has not influenced training or model selection in any way — `load_best_model_at_end` selected on **validation** macro-F1. These are the numbers to quote in the dissertation.

In [ ]:
test_metrics = trainer.evaluate(ds["test"], metric_key_prefix="test")
for k, v in test_metrics.items():
    if isinstance(v, float):
        print(f"{k:28s} {v:.4f}")

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix

preds_output = trainer.predict(ds["test"])
y_pred = np.argmax(preds_output.predictions, axis=-1)
y_true = preds_output.label_ids

report = classification_report(
    y_true, y_pred, target_names=labels, digits=3, zero_division=0
)
print(report)

with open("classification_report.txt", "w") as f:
    f.write(report)

In [ ]:
import matplotlib.pyplot as plt

cm = confusion_matrix(y_true, y_pred, normalize="true")
fig, ax = plt.subplots(figsize=(max(8, NUM_LABELS * 0.5), max(7, NUM_LABELS * 0.45)))
im = ax.imshow(cm, cmap="copper_r", vmin=0, vmax=1)
ax.set_xticks(range(NUM_LABELS), labels, rotation=90, fontsize=8)
ax.set_yticks(range(NUM_LABELS), labels, fontsize=8)
ax.set_xlabel("predicted")
ax.set_ylabel("actual")
ax.set_title("Normalised confusion matrix (test split)")
fig.colorbar(im, ax=ax, fraction=0.046)
plt.tight_layout()
plt.savefig("confusion_matrix.png", dpi=140)
plt.show()

In [ ]:
# The most-confused class pairs — genuinely useful discussion material for the
# evaluation chapter (e.g. Tshirts vs Tops is a real human-ambiguous boundary,
# not simply a model failure).
cm_counts = confusion_matrix(y_true, y_pred)
confusions = []
for i in range(NUM_LABELS):
    for j in range(NUM_LABELS):
        if i != j and cm_counts[i, j] > 0:
            confusions.append((cm_counts[i, j], labels[i], labels[j]))
confusions.sort(reverse=True)

print("Top 15 confusions (actual -> predicted):")
for count, actual, predicted in confusions[:15]:
    print(f"  {count:4d}  {actual}  ->  {predicted}")

## 6. Qualitative check — actual predictions on test images

In [ ]:
raw_test = load_dataset(DATASET_REPO_ID, split="test")  # untransformed, for display
rng = np.random.default_rng(RANDOM_SEED)
idxs = rng.choice(len(raw_test), size=12, replace=False)

fig, axes = plt.subplots(2, 6, figsize=(17, 7))
for ax, idx in zip(axes.flatten(), idxs):
    row = raw_test[int(idx)]
    actual = id2label[row["label"]]
    predicted = id2label[int(y_pred[int(idx)])]
    correct = actual == predicted
    ax.imshow(row["image"])
    ax.set_title(
        f"{'OK' if correct else 'MISS'}\nactual: {actual}\npred: {predicted}",
        fontsize=8,
        color="#3A6B3A" if correct else "#A65A4A",
    )
    ax.axis("off")
plt.tight_layout()
plt.savefig("sample_predictions.png", dpi=140)
plt.show()

## 7. Model card + push to the Hub

The model card is not decoration — an unlabelled weights file is not a research artefact. It records the task, training data, metrics, and (importantly) the model's **limitations**, which is what makes it citable in the dissertation.

In [ ]:
test_acc = test_metrics.get("test_accuracy", float("nan"))
test_macro_f1 = test_metrics.get("test_macro_f1", float("nan"))
test_weighted_f1 = test_metrics.get("test_weighted_f1", float("nan"))

model_card = f"""---
license: mit
base_model: {BASE_MODEL}
tags:
  - image-classification
  - fashion
  - vision
datasets:
  - {DATASET_REPO_ID}
metrics:
  - accuracy
  - f1
library_name: transformers
pipeline_tag: image-classification
---

# ThreadCraft Garment Classifier

Fine-tuned `{BASE_MODEL}` that predicts **garment type** from a clothing image.

Built for [ThreadCraft](https://github.com/Samandee-Galagoda/threadcraft), an AI-powered
custom clothing design and ordering platform, as a final-year BSc Software Engineering
project. In the product it runs on customer-uploaded reference photos at the design
wizard's Step 2, to suggest which cloth type the customer is describing.

## Intended use

Suggesting a garment category from a reference image, as an **assistive default** that the
customer can override — not as an authoritative classification.

## Results (held-out test split)

| Metric | Score |
|---|---|
| Accuracy | {test_acc:.4f} |
| Macro F1 | {test_macro_f1:.4f} |
| Weighted F1 | {test_weighted_f1:.4f} |

**Macro F1 is the headline number.** Accuracy is inflated by the dominant classes; macro F1
weights every garment type equally and so reflects performance on the rarer ones.

Classes ({NUM_LABELS}): {", ".join(labels)}

## Training

| | |
|---|---|
| Base model | `{BASE_MODEL}` |
| Dataset | [`{DATASET_REPO_ID}`](https://huggingface.co/datasets/{DATASET_REPO_ID}) |
| Train / Val / Test | {len(ds['train']):,} / {len(ds['validation']):,} / {len(ds['test']):,} |
| Epochs | {NUM_EPOCHS} |
| Batch size | {BATCH_SIZE} per device |
| Learning rate | {LEARNING_RATE} |
| Precision | fp16 |
| Hardware | Kaggle T4 x2 |
| Model selection | best validation macro-F1 |

## Limitations

- **Source images are 60x80 px**, upscaled to {image_size}x{image_size}. Fine visual detail
  (fabric texture, stitching, small trims) is simply not present in the training data, so the
  model distinguishes garment silhouette far better than garment detail.
- Trained on **catalogue product photography** (clean background, flat-lay or on-model studio
  shots). Accuracy on casual user-taken photos, worn garments at odd angles, or cluttered
  backgrounds will be lower — this is a train/serve distribution shift and is the main reason
  the prediction is surfaced as an overridable suggestion.
- The class set is **long-tailed**; rarer garment types have materially lower per-class recall.
  See the per-class breakdown in `classification_report.txt`.
- The underlying dataset is Indian-retail-sourced (Myntra), so garment style distribution is
  regionally skewed. That happens to suit ThreadCraft's South Asian market, but it does limit
  generalisation elsewhere.
- Some confusions reflect genuinely ambiguous human labels (e.g. Tshirts vs Tops) rather than
  model error.

## Usage

```python
from transformers import pipeline

clf = pipeline("image-classification", model="{MODEL_REPO_ID}")
clf("reference_photo.jpg")
```

## Citation of the source data

Aggarwal, P. (2019). *Fashion Product Images Dataset*. Kaggle.
Accessed via the `ashraq/fashion-product-images-small` mirror on the Hugging Face Hub.
"""

with open(f"{OUTPUT_DIR}/README.md", "w") as f:
    f.write(model_card)
print(model_card[:1800])

In [ ]:
import json

# Persist metrics as a file too, so the dissertation numbers survive even if the
# notebook output is lost.
metrics_blob = {
    "test": {k: float(v) for k, v in test_metrics.items() if isinstance(v, (int, float))},
    "num_labels": NUM_LABELS,
    "labels": labels,
    "base_model": BASE_MODEL,
    "epochs": NUM_EPOCHS,
    "batch_size": BATCH_SIZE,
    "learning_rate": LEARNING_RATE,
    "train_size": len(ds["train"]),
    "val_size": len(ds["validation"]),
    "test_size": len(ds["test"]),
}
with open(f"{OUTPUT_DIR}/metrics.json", "w") as f:
    json.dump(metrics_blob, f, indent=2)
print(json.dumps(metrics_blob["test"], indent=2))

In [ ]:
if PUSH_TO_HUB:
    from huggingface_hub import HfApi

    try:
        api = HfApi(token=HF_TOKEN)
        api.create_repo(MODEL_REPO_ID, exist_ok=True, repo_type="model")

        # Copy the evaluation figures in so they ship with the model card.
        import shutil

        for fname in [
            "confusion_matrix.png",
            "sample_predictions.png",
            "classification_report.txt",
        ]:
            if os.path.exists(fname):
                shutil.copy(fname, f"{OUTPUT_DIR}/{fname}")

        api.upload_folder(folder_path=OUTPUT_DIR, repo_id=MODEL_REPO_ID, repo_type="model")
        print(f"Pushed: https://huggingface.co/{MODEL_REPO_ID}")
    except Exception as e:
        print(f"PUSH FAILED: {e}")
        print(f"Weights are safe in {OUTPUT_DIR} — re-run just this cell to retry.")
else:
    print(f"PUSH_TO_HUB is False — model left in {OUTPUT_DIR}.")

## 8. Verify the pushed model actually loads and predicts

Never trust a push you haven't loaded back. This is what catches the classic failure of pushing the model but forgetting the image processor.

In [ ]:
if PUSH_TO_HUB:
    from transformers import pipeline

    try:
        clf = pipeline("image-classification", model=MODEL_REPO_ID, device=-1)
        test_img = raw_test[0]["image"]
        print(f"Actual: {id2label[raw_test[0]['label']]}")
        for pred in clf(test_img)[:3]:
            print(f"  {pred['label']:20s} {pred['score']:.4f}")
        print("\nRound-trip OK — the model + processor both loaded from the Hub.")
    except Exception as e:
        print(f"Round-trip FAILED: {e}")

## Summary for the dissertation

In [ ]:
print("=" * 62)
print("GARMENT CLASSIFIER — RESULTS")
print("=" * 62)
print(f"Base model      : {BASE_MODEL}")
print(f"Classes         : {NUM_LABELS}")
print(f"Train/Val/Test  : {len(ds['train']):,} / {len(ds['validation']):,} / {len(ds['test']):,}")
print(f"Epochs          : {NUM_EPOCHS}")
print("-" * 62)
print(f"Test accuracy   : {test_acc:.4f}")
print(f"Test macro F1   : {test_macro_f1:.4f}   <- headline metric")
print(f"Test weighted F1: {test_weighted_f1:.4f}")
print("=" * 62)
print(f"\nModel: https://huggingface.co/{MODEL_REPO_ID}")
print("Figures saved: confusion_matrix.png, sample_predictions.png, classification_report.txt")